In [ ]:
import pandas as pd

df = pd.read_csv("../../runs_kahan_final_final_export.csv")

# Simplify vacancy_mask to v1/v2
df["mask_version"] = df["vacancy_mask"].apply(
    lambda v: "v2" if "v2" in str(v) else ("v1" if pd.notna(v) and str(v).strip() else "")
)

# Auto-fill test_best_f2_computed for v2 runs where it's still empty
# (for v2, val=Brooklyn so best_val_f2 is the test metric)
if "test_best_f2_computed" not in df.columns:
    df["test_best_f2_computed"] = None
v2_missing = df["mask_version"].eq("v2") & df["test_best_f2_computed"].isna()
df.loc[v2_missing, "test_best_f2_computed"] = df.loc[v2_missing, "best_val_f2"]

# Drop columns that aren't needed for comparison
drop_cols = ["best_epoch", "lr", "min_vacant_pixels", "band_dropout_p", "seed",
             "vacancy_mask", "patch_splits", "trained_at", "machine"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Put test_best_f2_computed first
cols = ["test_best_f2_computed"] + [c for c in df.columns if c != "test_best_f2_computed"]
df = df[cols]

# Sort by test_best_f2_computed desc, then test_f2 desc
df = df.sort_values(
    ["test_best_f2_computed", "test_f2"],
    ascending=[False, False],
    na_position="last",
)

df

In [ ]:
df.to_csv("sorted_runs.csv")